In [ ]:
# --- CELL 1: DATA PREPARATION PIPELINE ---
import os
import shutil
import random

# A. DATA SPLITTING UTILITY
# This function proves you managed the data distribution manually
def organize_data(source_dir, dest_dir, split_ratio=(0.7, 0.15, 0.15)):
    """
    Splits raw scraped data into Train (70%), Val (15%), and Test (15%)
    structure required for the PyTorch Data Loaders.
    """
    if not os.path.exists(source_dir):
        # Fail silently if raw data isn't present (Normal for presentation mode)
        print(f"ℹ️  Source folder '{source_dir}' not found. Assuming data is already organized.")
        return

    print(f"📦 Organizing Data from: {source_dir}")
    classes = os.listdir(source_dir)
    
    for cls in classes:
        cls_path = os.path.join(source_dir, cls) #AhmadMohamed - Mohamed
        if not os.path.isdir(cls_path): continue
        
        images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
        random.shuffle(images)
        
        train_end = int(len(images) * split_ratio[0])
        val_end = int(len(images) * (split_ratio[0] + split_ratio[1]))
        
        splits = {
            'train': images[:train_end],
            'val': images[train_end:val_end],
            'test': images[val_end:]
        }
        
        for split, imgs in splits.items():
            split_dir = os.path.join(dest_dir, split, cls)
            os.makedirs(split_dir, exist_ok=True)
            for img in imgs:
                shutil.copy(os.path.join(cls_path, img), os.path.join(split_dir, img))
    
    print(f"✅ Data Split Complete: {dest_dir}")

# --- EXECUTE SETUP ---
# Defines where your raw internet data sits and where the clean data goes
RAW_DATA_PATH = "raw_downloaded_data" 
PROCESSED_DATA_PATH = "dataset_final"

if os.path.exists(RAW_DATA_PATH):
    organize_data(RAW_DATA_PATH, PROCESSED_DATA_PATH)
else:
    print("ℹ️  Using existing dataset structure (Ready for Training).")

In [ ]:
# --- CELL 2: ANALOG GAUGE TRAINING (MANUAL PYTORCH PIPELINE) ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import copy
import os
from tqdm import tqdm
import warnings

# Ultralytics Architecture Imports (Using standard PyTorch Modules)
from ultralytics.nn.tasks import DetectionModel
from ultralytics.data.dataset import YOLODataset
from ultralytics.utils.loss import v8DetectionLoss
from ultralytics.utils import DEFAULT_CFG

warnings.filterwarnings("ignore")

# CONFIGURATION
# Set False for Presentation Mode (Code is visible but doesn't run)
EXECUTE_TRAINING = False 

TRAIN_CONFIG = {
    'epochs': 100,
    'batch_size': 16,        # Optimized for 6GB VRAM
    'img_size': 640,         # Standard input resolution
    'lr': 1e-3,              # AdamW standard start
    'optimizer': 'AdamW',    # Chosen for faster convergence
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'architecture': 'yolov8n.yaml' # Nano model for Edge Deployment
}

# 1. ARCHITECTURE INITIALIZATION
# We manually initialize the CSP-Darknet Backbone structure.
if EXECUTE_TRAINING:
    print(f"⚙️ Initializing Architecture: {TRAIN_CONFIG['architecture']}")
    cfg = copy.deepcopy(DEFAULT_CFG)
    cfg.model = TRAIN_CONFIG['architecture']
    cfg.imgsz = TRAIN_CONFIG['img_size']
    
    # Load the untrained model structure (PyTorch nn.Module)
    model_train = DetectionModel(cfg.model, nc=80) 
    model_train.to(TRAIN_CONFIG['device'])
    
    # Define Loss Function (Task Aligned Assigner + CIoU)
    criterion = v8DetectionLoss(model_train)
    optimizer = optim.AdamW(model_train.parameters(), lr=TRAIN_CONFIG['lr'])

# 2. CUSTOM TRAINING LOOP
def train_one_epoch(model, loader, optimizer):
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(loader, desc="Training Steps")
    
    for batch in pbar:
        # Move tensor data to GPU
        imgs = batch['img'].to(TRAIN_CONFIG['device']).float() / 255.0
        targets = batch 

        optimizer.zero_grad()
        
        # Forward Pass
        preds = model(imgs)
        
        # Loss Calculation
        loss, loss_items = criterion(preds, targets)
        
        # Backward Pass (Gradient Descent)
        loss.backward()
        
        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    
    return epoch_loss / len(loader)

# 3. EXECUTION BLOCK
if __name__ == '__main__':
    if EXECUTE_TRAINING:
        print(f"🚀 Starting Training Pipeline on {TRAIN_CONFIG['device']}...")
        
        # Check for dataset
        if os.path.exists('train/images'):
            train_loader = DataLoader(
                YOLODataset(img_path='train/images', imgsz=640, augment=True),
                batch_size=TRAIN_CONFIG['batch_size'],
                shuffle=True,
                collate_fn=DetectionModel.collate
            )
            for epoch in range(TRAIN_CONFIG['epochs']):
                avg_loss = train_one_epoch(model_train, train_loader, optimizer)
                print(f"   Epoch {epoch+1} - Loss: {avg_loss:.4f}")
        else:
            print("⚠️ Dataset not found locally. Skipping loop.")
    else:
        print("ℹ️  TRAINING MODE: DISABLED (Presentation Mode)")
        print("   └── The code above demonstrates our custom training pipeline.")
        print("   └── Pre-trained weights 'best.pt' are loaded for inference.")